# Import statements

In [265]:
# import statements

# astropy
from astropy.io import fits
from astropy.wcs import WCS
import astropy.units as u
from astropy.table import QTable, Table
from astropy import constants as const
from astropy.coordinates import SkyCoord

from photutils.aperture import SkyCircularAperture, ApertureStats

# matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# general / misc
import numpy as np
import pandas as pd
import sys
from reproject import reproject_interp

In [264]:
print(w1_7p5_map.unit)

MJY/SR


# Fetching galaxy properties from PHANGS table

In [136]:
# read in the PHANGS sample table -- note that public version with basic info doesn't have inclination, which we need
props_tab=Table.read('/Users/adignan/Documents/GitHub/co_to_h2/phangs_sample_table_v1p6.csv',header_start=184)

# pick our sample ahead of time
targets=['ngc0628', 'ngc1097', 'ngc3351', 'ngc3521', 'ngc3621', 'ngc3627',
         'ngc4254', 'ngc4321', 'ngc4536', 'ngc4569', 'ngc4579', 'ngc4631',
         'ngc4826', 'ngc5194', 'ngc5457', 'ngc7793']

props_filt=props_tab[np.isin(props_tab['name'], targets)]
# props_filt

# Required files

In [258]:
# CO moment zero map and corresponding uncertainty map
# WISE 1, 3, and 4 maps 
co='/Users/adignan/research/phangs/ngc0628/alma/ngc0628_12m+7m+tp_co21_2.566192048arcsec_broad_mom0.fits'
co_err='/Users/adignan/research/phangs/ngc0628/alma/ngc0628_12m+7m+tp_co21_2.566192048arcsec_broad_emom0.fits'

w1_7p5='/Users/adignan/research/phangs/wise_all/ngc0628_w1_gauss7p5.fits'
w1_15='/Users/adignan/research/phangs/wise_all/ngc0628_w1_gauss15.fits'
w3_7p5='/Users/adignan/research/phangs/wise_all/ngc0628_w3_gauss7p5.fits'
w4_15='/Users/adignan/research/phangs/wise_all/ngc0628_w4_gauss15.fits'

# comparisons
massmap='/Users/adignan/Documents/GitHub/co_to_h2/NGC0628.ica2.massmap.sm_Msunpc2.fits'

# Defining other functions

In [259]:
def plotmap(data_in, clabel=None, norm=False, cmap='viridis', title=None):
    plt.figure()
    if norm:
        plt.imshow(data_in, norm=LogNorm(), cmap=cmap)
    else:
        plt.imshow(data_in, cmap=cmap)
    if title:
        plt.title(title)
    if clabel:
        plt.colorbar(label=clabel)
    if clabel==None:
        plt.colorbar()

def photometry(data, ra_in, dec_in, radius, wcs, deg=True):

    if deg==False:
        r_split=ra_in.split(':')
        ra=(r_split[0]+'h'+r_split[1]+'m'+r_split[2]+'s')

        d_split=dec_in.split(':')
        dec=(d_split[0]+'d'+d_split[1]+'m'+d_split[2]+'s')
    else:
        ra=ra_in*u.deg
        dec=dec_in*u.deg

    pos=SkyCoord(ra, dec)
    aperture=SkyCircularAperture(pos, radius)
    aperstats= ApertureStats(data=data, aperture=aperture, wcs=wcs)
    return aperstats.sum

# Defining class and methods

In [269]:
class Map:
    '''This class sets up a FITS file for calculations.'''

    def __init__(self, path=None, data=None, header=None, unit=None, wcs=None):
        '''Extracts the data and header from the FITS file.'''
        # extract data array and header from FITS file
        if path is not None:
            data, header = fits.getdata(path, ext=0, header=True)
            self.hdr = header
            self.wcs = WCS(self.hdr)

        # split into attributes 
        self.data = data
        self.hdr = header

        if unit:
            self.unit = unit
        elif header is not None and header['BUNIT'] is not None:
            if header['BUNIT']=='MJY/SR':
                unit=u.MJy/u.sr 
            else:
                unit=u.Unit(header['BUNIT'])
            self.unit = unit
        if unit is None:
            self.unit = u.dimensionless_unscaled

    def reproject(self, template_file, plot=False, title=None):
        '''Reprojects the FITS file data onto the WCS and size of the input
        template FITS file and returns the reprojected data array as 
        a Quantity based on the original FITS file data units.'''
        # create an instance of the input template FITS file
        template=Map(template_file)

        # remove all axes with length=1 so array is 2D
        data_in = np.squeeze(self.data)

        # get WCS for file to be reprojected and template 
        wcs_in=WCS(self.hdr)
        wcs_out=WCS(template.hdr)

        # get size for reprojection from header of template
        ny=template.hdr['NAXIS2']
        nx=template.hdr['NAXIS1']

        # reproject!
        data_out, footprint = reproject_interp((data_in, wcs_in), wcs_out,
                                               shape_out=(ny, nx))
        # replace any invalid values in reprojected image with NaN
        data_out[footprint <= 0] = np.nan

        if plot == True:
            plotmap(data_out, norm=True, clabel=str(self.unit), title=title)

        return Map(data=data_out, header=self.hdr)

    def add_col(self, t, name, values=None, length=None):
        '''Adds a column called 'name' to a table t.
        Flattens the data and adds as a Quantity.'''
        if values is None:
            values=self.data
            unit = self.unit
        else:
            if hasattr(values, 'unit'):
                unit=values.unit
                values=values.value
        
        if length:
            t[name] = np.full(length, values) * unit
        else:
            t[name] = np.array(values).reshape(-1) * unit
        return t[name]
    
    def calc_gamma(self, method='', gal_sfr = None, gal_Mstar = None,
               I_w4 = None, I_w3 = None, I_w1 = None):
        # All colors are logarithmic ratios of specific luminosity in Jy, i.e., Lν and not νLν (Leroy+19).
        if method == 'GSWLC':
            # maybe put in a "if gal_sfr and gal_Mstar blank then print need that stuff"
            a = -10.9
            b = -0.21
            c = -9.5
            gal_sfr = gal_sfr * u.M_sun / u.yr
            gal_Mstar = gal_Mstar * u.M_sun
            q=np.log10(gal_sfr.value/gal_Mstar.value)
            
        if method == 'W3/W1':
            a = 0.1
            b = -0.46
            c = 0.75
            q=np.log10(I_w3.value/I_w1.value)

        if method == 'W4/W1':
            a = 0.0
            b = -0.4
            c = 0.75
            q=np.log10(I_w4/I_w1).value

        gamma = np.where(q < a, 0.5,
                np.where(q > c, 0.2,
                0.5+b*(q - a)))
        
        if method == 'GSWLC':
            attr_name = 'gam_gswlc'
        if method == 'W3/W1':
            attr_name = 'gam_w3w1'
        if method == 'W4/W1':
            attr_name = 'gam_w4w1'

        setattr(self, attr_name, gamma * u.M_sun / u.L_sun)

        return getattr(self, attr_name)
    
    def calc_sig_star(self, gamma, I_w1, i, plot=False, method=None):
        sig_star = 330 * (gamma/0.5) * I_w1/(u.MJy/u.sr) * np.cos(np.deg2rad(i))

        if hasattr(self, 'gam_gswlc'):
            attr_name = 'sigstar_gswlc'
        if hasattr(self, 'gam_w3w1'):
            attr_name = 'sigstar_w3w1'
        if hasattr(self, 'gam_w4w1'):
            attr_name = 'sigstar_w4w1'

        if plot == True:
            plotmap(sig_star.data, norm=True, clabel=r'M$_\odot$/pc$^{2}$', cmap='inferno', 
                    title=rf'$\Sigma_{{\star}}: \mathrm{{{method}}}$')
    
        setattr(self, attr_name, sig_star)
        return getattr(self, attr_name)

    # code from https://github.com/astrojysun/COConversionFactor
    def deproject(self, center_coord=None, incl=0*u.deg, pa=0*u.deg,
                header=None, wcs=None, naxis=None, ra=None, dec=None,
                return_offset=False, linear=True, distance=None):

        """
        Calculate deprojected radii and projected angles in a disk.

        This function deals with projected images of astronomical objects
        with an intrinsic disk geometry. Given sky coordinates of the
        disk center, disk inclination and position angle, this function
        calculates deprojected radii and projected angles based on
        (1) a FITS header (`header`), or
        (2) a WCS object with specified axis sizes (`wcs` + `naxis`), or
        (3) RA and DEC coodinates (`ra` + `dec`).
        Both deprojected radii and projected angles are defined relative
        to the center in the inclined disk frame. For (1) and (2), the
        outputs are 2D images; for (3), the outputs are arrays with shapes
        matching the broadcasted shape of `ra` and `dec`.

        Parameters
        ----------
        center_coord : `~astropy.coordinates.SkyCoord` object or 2-tuple
            Sky coordinates of the disk center
        incl : `~astropy.units.Quantity` object or number, optional
            Inclination angle of the disk (0 degree means face-on)
            Default is 0 degree.
        pa : `~astropy.units.Quantity` object or number, optional
            Position angle of the disk (red/receding side, North->East)
            Default is 0 degree.
        header : `~astropy.io.fits.Header` object, optional
            FITS header specifying the WCS and size of the output 2D maps
        wcs : `~astropy.wcs.WCS` object, optional
            WCS of the output 2D maps
        naxis : array-like (with two elements), optional
            Size of the output 2D maps
        ra : array-like, optional
            RA coordinate of the sky locations of interest
        dec : array-like, optional
            DEC coordinate of the sky locations of interest
        return_offset : bool, optional
            Whether to return the angular offset coordinates together with
            deprojected radii and angles. Default is to not return.

        Returns
        -------
        deprojected coordinates : list of arrays
            If `return_offset` is set to True, the returned arrays include
            deprojected radii, projected angles, as well as angular offset
            coordinates along East-West and North-South direction;
            otherwise only the former two arrays will be returned.

        Notes
        -----
        This is the Python version of an IDL function `deproject` included
        in the `cpropstoo` package. See URL below:
        https://github.com/akleroy/cpropstoo/blob/master/cubes/deproject.pro
        """

        if isinstance(center_coord, SkyCoord):
            x0_deg = center_coord.ra.degree
            y0_deg = center_coord.dec.degree
        else:
            x0_deg, y0_deg = center_coord
            if hasattr(x0_deg, 'unit'):
                x0_deg = x0_deg.to(u.deg).value
                y0_deg = y0_deg.to(u.deg).value
        if hasattr(incl, 'unit'):
            incl_deg = incl.to(u.deg).value
        else:
            incl_deg = incl
        if hasattr(pa, 'unit'):
            pa_deg = pa.to(u.deg).value
        else:
            pa_deg = pa

        if header is not None:
            wcs_cel = WCS(header).celestial
            naxis1 = header['NAXIS1']
            naxis2 = header['NAXIS2']
            # create ra and dec grids
            ix = np.arange(naxis1)
            iy = np.arange(naxis2).reshape(-1, 1)
            ra_deg, dec_deg = wcs_cel.wcs_pix2world(ix, iy, 0)
        elif (wcs is not None) and (naxis is not None):
            wcs_cel = wcs.celestial
            naxis1, naxis2 = naxis
            # create ra and dec grids
            ix = np.arange(naxis1)
            iy = np.arange(naxis2).reshape(-1, 1)
            ra_deg, dec_deg = wcs_cel.wcs_pix2world(ix, iy, 0)
        else:
            ra_deg, dec_deg = np.broadcast_arrays(ra, dec)
            if hasattr(ra_deg, 'unit'):
                ra_deg = ra_deg.to(u.deg).value
                dec_deg = dec_deg.to(u.deg).value

        # recast the ra and dec arrays in term of the center coordinates
        # arrays are now in degrees from the center
        dx_deg = (ra_deg - x0_deg) * np.cos(np.deg2rad(y0_deg))
        dy_deg = dec_deg - y0_deg

        # rotation angle (rotate x-axis up to the major axis)
        rotangle = np.pi/2 - np.deg2rad(pa_deg)

        # create deprojected coordinate grids
        deprojdx_deg = (dx_deg * np.cos(rotangle) +
                        dy_deg * np.sin(rotangle))
        deprojdy_deg = (dy_deg * np.cos(rotangle) -
                        dx_deg * np.sin(rotangle))
        deprojdy_deg /= np.cos(np.deg2rad(incl_deg))

        # make map of deprojected distance from the center
        radius_deg = np.sqrt(deprojdx_deg**2 + deprojdy_deg**2)

        # make map of angle w.r.t. position angle
        projang_deg = np.rad2deg(np.arctan2(deprojdy_deg, deprojdx_deg))

        self.projang_deg=projang_deg
        self.dx_deg=dx_deg
        self.dy_deg=dy_deg

        if linear==True and distance is None:
            raise ValueError("Please provide a distance with units.")
        if linear==True and distance is not None:
            if hasattr(distance, 'unit'):
                distance_kpc=(distance).to(u.kpc).value 
            if not hasattr(distance, 'unit'):
                raise ValueError("Please give units with the distance.")
            
            r_g = radius_deg * u.deg.to(u.radian) * distance_kpc
            r_g_kpc = r_g * u.kpc

            self.radius_kpc=r_g_kpc
            return self.radius_kpc

        if linear==False:

            self.radius_deg=radius_deg

            if return_offset:
                return self.radius_deg, self.projang_deg, self.dx_deg, self.dy_deg
            else:
                return self.radius_deg, self.projang_deg
        
    # this is also from Jiayi Sun
    
    # this function is needed for calc_metallicity
    def predict_logOH_SAMI19(self,
            Mstar, calibration='PP04', form='MZR', return_residual=False):
        """
        Predict 12+log(O/H) with the 'SAMI19' MZR (Sanchez+19).

        This function predicts the gas phase abundance 12+log(O/H)
        from the global stellar mass of a galaxy according to the
        mass-metallicity relations reported in Sanchez+19.
        Reference: Sanchez et al. (2019), MNRAS, 484, 3042

        Parameters
        ----------
        Mstar : number, `~numpy.ndarray`, `~astropy.units.Quantity` object
            Galaxy global stellar mass, in units of solar mass
        calibration : {'O3N2-M13', 'PP04', 'N2-M13', 'ONS', 'R23', ...}
            Metallicity calibration to adopt (see Table 1 in Sanchez+19).
            Default is 'PP04'.
        form : {'MZR', 'pMZR'}
            The MZR functional form to adopt (see Table 1 in Sanchez+19).
            Default is 'MZR'.
        return_residual : bool
            Whether to return the residual scatter around the MZR.
            Default is to not return.

        Returns
        -------
        logOH : number or `~numpy.ndarray`
            Predicted gas phase abundance, in units of 12+log(O/H)
        """

        calibrations = [
            'O3N2-M13', 'PP04', 'N2-M13', 'ONS', 'R23', 'pyqz', 't2',
            'M08', 'T04', 'EPM09', 'DOP16']
        if calibration not in calibrations:
            raise ValueError(
                "Available choices for `calibration` are: "
                "{}".format(calibrations))

        if hasattr(Mstar, 'unit'):
            x = np.log10(Mstar.to(u.Msun).value) - 8
        else:
            x = np.log10(Mstar) - 8

        # Mass-metallicity relation
        if form == 'MZR':  # MZR best fit
            params = np.zeros(
                3, dtype=[(calib, 'f') for calib in calibrations])
            params[0] = (  # a
                8.51, 8.73, 8.50, 8.51, 8.48, 9.01, 8.84,
                8.88, 8.84, 8.54, 8.94)
            params[1] = (  # b
                0.007, 0.010, 0.008, 0.011, 0.004, 0.017, 0.008,
                0.010, 0.007, 0.002, 0.020)
            params[2] = (  # sigma_MZR
                0.102, 0.147, 0.105, 0.138, 0.101, 0.211, 0.115,
                0.169, 0.146, 0.074, 0.288)
            a = params[calibration][0]
            b = params[calibration][1]
            c = 3.5
            residual = params[calibration][2]
            logOH = a + b*(x-c)*np.exp(-(x-c))

        elif form == 'pMZR':  # pMZR polynomial fit
            params = np.zeros(
                5, dtype=[(calib, 'f') for calib in calibrations])
            params[0] = (  # p0
                8.478, 8.707, 8.251, 8.250, 8.642, 8.647, 8.720,
                8.524, 8.691, 8.456, 8.666)
            params[1] = (  # p1
                -0.529, -0.797, -0.207, -0.428, -0.589, -0.718, -0.487,
                -0.148, -0.200, -0.097, -0.991)
            params[2] = (  # p2
                0.409, 0.610, 0.243, 0.427, 0.370, 0.682, 0.415,
                0.218, 0.164, 0.130, 0.738)
            params[3] = (  # p3
                -0.076, -0.113, -0.048, -0.086, -0.063, -0.133, -0.080,
                -0.040, -0.023, -0.032, -0.114)
            params[4] = (  # sigma_pMZR
                0.077, 0.112, 0.078, 0.101, 0.087, 0.143, 0.087,
                0.146, 0.123, 0.071, 0.207)
            p0 = params[calibration][0]
            p1 = params[calibration][1]
            p2 = params[calibration][2]
            p3 = params[calibration][3]
            residual = params[calibration][4]
            logOH = p0 + p1*x + p2*x**2 + p3*x**3

        else:
            raise ValueError("Invalid input value for `form`!")

        self.logOH=logOH
        self.residual=residual

        if return_residual:
            return self.logOH, self.residual
        else:
            return self.logOH

    # also needed for calc_metallicity
    def extrapolate_logOH_radially(self,
            logOH_Re, gradient='CALIFA14', Rgal=None, Re=None):
        """
        Extrapolate 12+log(O/H) assuming a fixed radial gradient.

        This function extrapolates the gas phase abundance 12+log(O/H)
        from its value at 1 Re to the entire galaxy, according to a fixed
        radial gradient specified by the user.

        Parameters
        ----------
        logOH_Re : number, `~numpy.ndarray`
            Gas phase abundance at 1 Re, in units of 12+log(O/H)
        gradient : {'CALIFA14', float}
            Radial abundance gradient to adopt, in units of dex/Re.
            Default is 'CALIFA14', i.e., -0.10 dex/Re
            (Reference: Sanchez et al. 2014, A&A, 563, A49).
        Rgal : number, ndarray, Quantity object
            Galactocentric radii, in units of kilo-parsec
        Re : number, ndarray, Quantity object
            Galaxy effective radii, in units of kilo-parsec

        Returns
        -------
        logOH : number or `~numpy.ndarray`
            Predicted gas phase abundance, in units of 12+log(O/H)
        """

        if (Rgal is None) or (Re is None):
            return logOH_Re
        
        if hasattr(Rgal, 'unit') and hasattr(Re, 'unit'):
            Rgal_normalized = (Rgal / Re).to('').value
        elif hasattr(Rgal, 'unit') or hasattr(Re, 'unit'):
            raise ValueError(
                "`Rgal` and `Re` should both carry units "
                "or both be dimensionless")
        else:
            Rgal_normalized = np.asarray(Rgal) / np.asarray(Re)

        # metallicity gradient
        if gradient == 'CALIFA14':
            alpha_logOH = -0.10  # dex/Re
            logOH = (logOH_Re + alpha_logOH * (Rgal_normalized - 1))
        else:
            alpha_logOH = gradient  # dex/Re
            logOH = (logOH_Re + alpha_logOH * (Rgal_normalized - 1))

        self.logOH=logOH
        return self.logOH

    def calc_metallicity(self, colname='Zprime', unit='',
            Mstar=None, Re=None, r_gal=None,
            logOH_solar=None):
        """ 
        Calculate the metallicity based on the method specified.

        Parameters:
        ----------
        colname : str
            Name of the column to store the metallicity.
        unit : str
            Unit of the metallicity.
        method : str
            Method to use for the calculation.
        Mstar : Quantity
            Stellar mass.
        Rstar : Quantity
            Stellar radius.
        r_gal : Quantity
            Galactic radius.
        logOH_solar : float
            Solar metallicity.

        """

        from CO_conversion_factor import metallicity
        import astropy.units as u

        if logOH_solar is None:
            logOH_solar = 8.69  # Asplund+09

        logOH_Re = self.predict_logOH_SAMI19(
            Mstar * 10**0.10)  # Fig. A1 in Sanchez+19
        logOH = self.extrapolate_logOH_radially(
            logOH_Re, gradient='CALIFA14',
            Rgal=r_gal, Re=Re)  # Eq. A.3 in Sanchez+14
        
        Zprime = (10 ** (logOH - logOH_solar) * u.Unit('')).to(unit)
        self.Zprime=Zprime
        return self.Zprime

    def calc_alpha_co(self, Zprime, sigma_star, Zprime_ll=0.2, Zprime_ul=2.0, 
                        sigma_star_thresh=100, sigma_star_ul=np.inf,
                        J='2-1'):
        alpha_co_mw=4.35 * (u.M_sun/(u.pc)**2) * (1/ ((u.K*u.km)/u.s) )
        Z_term=np.clip(Zprime, Zprime_ll, Zprime_ul)**(-1.5)
        sb_term=(np.clip(sigma_star, sigma_star_thresh, sigma_star_ul)/sigma_star_thresh)**(-0.25)
        if J=='2-1':
            rco=0.65
        if J=='1-0':
            rco=1

        alpha_co=alpha_co_mw * Z_term * sb_term / rco
        self.alpha_co=alpha_co
        return self.alpha_co

    def alpha_to_x(self, alpha_co):
        x_co = alpha_co.cgs / (2.8*const.m_p.cgs)
        x_co = x_co.to(1 / (u.cm**2 * (u.K * u.km / u.s)))

        self.x_co=x_co
        return self.x_co

    def calc_nh2(self, x_co, i_co, inc):
        i_co = i_co*(u.K*u.km)/u.s # only do this if no units
        inc = inc*u.deg
        nh2 =  x_co * i_co * np.cos(np.deg2rad(inc))
        self.nh2 = nh2
        return self.nh2

In [256]:
ngc0628_sfr=props_filt['props_sfr'][0]
ngc0628_mstar=props_filt['props_mstar'][0]
ngc0628_inc=props_filt['orient_incl'][0]
ngc0628_pa=props_filt['orient_posang'][0]
ngc0628_dist=props_filt['dist'][0]

print(ngc0628_sfr,ngc0628_mstar)

1.7524408 21940600000.0


In [ ]:
w1_7p5_map=Map(w1_7p5)
w1_15_map=Map(w1_15)
w3_7p5_map=Map(w3_7p5)
w4_15_map=Map(w4_15)

co_map=Map(co)

mass_map=Map(massmap)
mass_map_reproj=mass_map.reproject(template_file=co)

print(np.min(w1_7p5_map.data))

w1_7p5_reproj=w1_7p5_map.reproject(template_file=co)
w1_15_reproj=w1_15_map.reproject(template_file=co)
w3_7p5_reproj=w3_7p5_map.reproject(template_file=co)
w4_15_reproj=w4_15_map.reproject(template_file=co)

# plotmap(w1_7p5_map.data)
# plotmap(w1_reproj.data)
# plotmap(co_map.data)

tab=QTable() # empty

w1_7p5_reproj.add_col(t=tab, name='reproj_w1_7p5')
w3_7p5_reproj.add_col(t=tab, name='reproj_w3_7p5')

# GSWLC 
w1_7p5_reproj.calc_gamma(method='GSWLC',gal_sfr=ngc0628_sfr,gal_Mstar=ngc0628_mstar) # should be scalar
w1_7p5_reproj.add_col(t=tab, name='GSWLC gamma', values=w1_7p5_reproj.gam_gswlc)

w1_7p5_reproj.calc_sig_star(gamma=w1_7p5_reproj.gam_gswlc, 
                                      I_w1=w1_7p5_reproj.data*w1_7p5_reproj.unit, i=ngc0628_inc)
w1_7p5_reproj.add_col(t=tab, name='GSWLC sigma star', values=w1_7p5_reproj.sigstar_gswlc)


# W3/W1
w1_7p5_reproj.calc_gamma(method='W3/W1', I_w3=w3_7p5_reproj.data*w3_7p5_reproj.unit, 
                                                    I_w1=w1_7p5_reproj.data*w1_7p5_reproj.unit) 
w1_7p5_reproj.add_col(t=tab, name='W3/W1 gamma', values=w1_7p5_reproj.gam_w3w1)

w1_7p5_reproj.calc_sig_star(gamma=w1_7p5_reproj.gam_w3w1, 
                                     I_w1=w1_7p5_reproj.data*w1_7p5_reproj.unit, i=ngc0628_inc)
w1_7p5_reproj.add_col(t=tab, name='W3/W1 sigma star', values=w1_7p5_reproj.sigstar_w3w1)

# W4/W1
w1_15_reproj.calc_gamma(method='W4/W1', I_w4=w4_15_reproj.data*w1_15_reproj.unit, 
                                                    I_w1=w1_15_reproj.data*w1_15_reproj.unit) 
w1_15_reproj.add_col(t=tab, name='W4/W1 gamma', values=w1_15_reproj.gam_w4w1)

w1_15_reproj.calc_sig_star(gamma=w1_15_reproj.gam_w4w1, I_w1=w1_15_reproj.data*w1_15_reproj.unit, 
                           i=ngc0628_inc, method='W4/W1')
w1_15_reproj.add_col(t=tab, name='W4/W1 sigma star', values=w1_15_reproj.sigstar_w4w1)

print(tab)

sys.exit()

# calculate deprojected radii grid 
co_map.deproject(center_coord=(co_map.hdr['CRVAL1']*u.deg, co_map.hdr['CRVAL2']*u.deg),
                    incl=ngc0628_inc*u.deg, pa=ngc0628_pa*u.deg, header=co_map.hdr, wcs=WCS(co_map.hdr),
                    naxis=np.array([co_map.hdr['NAXIS1'],co_map.hdr['NAXIS2']]),
                    linear=True,distance=ngc0628_dist*u.Mpc)

co_map.add_col(t=tab, name='r_G_kpc', values=co_map.radius_kpc)

co_map.calc_metallicity(Mstar=ngc0628_mstar, Re=3.9*u.kpc, r_gal=co_map.radius_kpc)
co_map.add_col(t=tab, name='Zprime', values=co_map.Zprime)
plotmap(co_map.Zprime,title='metallicity')

co_map.calc_alpha_co(Zprime=co_map.Zprime,sigma_star=w1_7p5_reproj.sig_star)
co_map.add_col(t=tab, name='alpha_co', values=co_map.alpha_co)

# convert data to K km / s * (pc^2 / pix)
hdr_new=co_map.hdr

out_wcs = WCS(co_map.hdr)
hdr_new = out_wcs.to_header()
hdr_new['BUNIT']='Msun pc-2 (K km s-1)^-1'

fits.writeto('/Users/adignan/research/phangs/alpha_co.fits',data=co_map.alpha_co.reshape(1600,1600).value,header=hdr_new,overwrite=True)

co_map.add_col(t=tab,name='CO data')

# converting mom0 map (K km / s) to hybrid (K km / s * pc^2 / pix)
pix_size_rad = (co_map.hdr['CDELT1']*u.deg).to(u.radian)

pix_size_lin = (pix_size_rad.value * (ngc0628_dist*u.Mpc).to(u.pc))**2

l_co_map = co_map.data*co_map.unit * pix_size_lin
m_mol=co_map.alpha_co*l_co_map.reshape(-1)
tab['M_mol']=m_mol
m_mol_map=Map(data=m_mol.reshape(1600,1600))

plotmap(m_mol.reshape(1600,1600),title='total mass')

phot=photometry(data=m_mol_map.data, ra_in='01:36:41.7', dec_in='15:46:59', radius= 7.7*u.arcsec, wcs=co_map.wcs, deg=False)



INFO: 
                Inconsistent SIP distortion information is present in the FITS header and the WCS object:
                SIP coefficients were detected, but CTYPE is missing a "-SIP" suffix.
                astropy.wcs is using the SIP distortion coefficients,
                therefore the coordinates calculated here might be incorrect.

                If you do not want to apply the SIP distortion coefficients,
                please remove the SIP coefficients from the FITS header or the
                WCS object.  As an example, if the image is already distortion-corrected
                (e.g., drizzled) then distortion components should not apply and the SIP
                coefficients should be removed.

                While the SIP distortion coefficients are being applied here, if that was indeed the intent,
                for consistency please append "-SIP" to the CTYPE in the FITS header or the WCS object.

                 [astropy.wcs.wcs]
INFO: 
             

In [ ]:
tab

reproj_w1_7p5,reproj_w3_7p5,GSWLC gamma,GSWLC sigma star,W3/W1 gamma,W3/W1 sigma star,W4/W1 gamma,W4/W1 sigma star
MJy / sr,MJy / sr,solMass / solLum,solMass2 / (solLum pc2),solMass / solLum,solMass2 / (solLum pc2),solMass / solLum,solMass2 / (solLum pc2)
float64,float64,float64,float64,float64,float64,float64,float64
0.014625113007629496,-0.06391425659342381,0.33149708025585223,3.1612742668655853,nan,nan,0.2,1.9103074590844091
0.014529883253856082,-0.06668496199598128,0.33149708025585223,3.1406899903620946,nan,nan,0.2,1.9114907610570087
0.014434653513140522,-0.06945566610495564,0.33149708025585223,3.1201057166811093,nan,nan,0.2,1.9126740633079202
0.01433942378541201,-0.07222636892033756,0.33149708025585223,3.0995214458073246,nan,nan,0.2,1.9138573658334004
0.014244194070828633,-0.07499707044365254,0.33149708025585223,3.078937177774911,nan,nan,0.2,1.9150406686419017
0.014148964369166243,-0.07776777067409885,0.33149708025585223,3.058352912535418,nan,nan,0.2,1.9162239717216405
0.014053734680454687,-0.08053846961302744,0.33149708025585223,3.0377686500952983,nan,nan,0.2,1.91740727507517
0.01395850500486898,-0.0833091672623083,0.33149708025585223,3.01718439049238,nan,nan,0.2,1.9185905787132087


In [219]:
# tab.write('/Users/adignan/research/phangs/table.csv', format='ascii.csv', overwrite=True)
tab[1280000]
tab_filt = tab[~np.isnan(tab['CO data'])]
tab_filt
print(np.median(tab_filt['alpha_co']))
print(co_map.radius_kpc)

6.742437154813266 K km / s
[[10.94730222 10.94045836 10.93361871 ... 10.84299192 10.84977464
  10.85656168]
 [10.94052199 10.93367394 10.92683011 ... 10.83625515 10.84304215
  10.84983347]
 [10.93374591 10.92689368 10.92004566 ... 10.82952263 10.83631391
  10.84310952]
 ...
 [10.75073554 10.74387309 10.73701492 ... 10.81990207 10.82681118
  10.83372447]
 [10.75739561 10.75053748 10.74368363 ... 10.82662852 10.83353341
  10.84044248]
 [10.76406005 10.75720623 10.7503567  ... 10.83335922 10.84025989
  10.84716475]] kpc


In [196]:
phot

<Quantity 3949968.43166119 solMass>

In [209]:
print((co_map.data))

[[nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 ...
 [nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]]


# Read in required FITS files

# Reproject WISE maps onto CO grid